### Dim_RoadSegment

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_road_segment AS target
USING (
    SELECT
        road_name, FIRST(frc) AS road_category_id, FIRST(length_of_road_section) AS length_of_road_segment_km,
        CASE 
            WHEN FIRST(frc) = 'FRC0' THEN 'Najvažniji putevi - autoputevi / glavne saobraćajnice'
            WHEN FIRST(frc) = 'FRC1' THEN 'Glavni putevi / važni gradski koridori'
            WHEN FIRST(frc) = 'FRC2' THEN 'Značajni regionalni / gradski putevi'
            WHEN FIRST(frc) = 'FRC3' THEN 'Ostali važniji gradski putevi'
            WHEN FIRST(frc) = 'FRC4' THEN 'Lokalni putevi'
            WHEN FIRST(frc) = 'FRC5' THEN 'Manji lokalni putevi' 
        END AS road_category,
        FIRST(coordinates) AS coordinates
    FROM bg_traffic.bg_traffic_silver.tomtom_flow
    WHERE road_name IS NOT NULL
    GROUP BY road_name
) AS source 
ON target.road_name = source.road_name
WHEN MATCHED THEN UPDATE SET
    target.road_category_id = source.road_category_id,
    target.length_of_road_segment_km = source.length_of_road_segment_km,
    target.road_category = source.road_category,
    target.coordinates = source.coordinates
WHEN NOT MATCHED THEN INSERT (
    road_segment_sid,
    road_name,
    road_category_id,
    length_of_road_segment_km,
    road_category,
    coordinates
)
VALUES (
    (SELECT COALESCE(MAX(road_segment_sid), 0) + 1
     FROM bg_traffic.bg_traffic_gold.dim_road_segment),
    source.road_name,
    source.road_category_id,
    source.length_of_road_segment_km,
    source.road_category,
    source.coordinates
);


### Dim_GTFS_Route

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_gtfs_route AS target
USING (
    SELECT
        r.route_id,
        r.route_short_name,
        r.route_long_name,
        r.route_type AS route_type_id,
        CASE
            WHEN r.route_type = 0 THEN 'Tramvaj'
            WHEN r.route_type = 3 THEN 'Autobus'
            WHEN r.route_type = 11 THEN 'Trolejbus'
            WHEN r.route_type = 2 THEN 'Železnica'
        END AS route_type_name,
        d.total_distance_km
    FROM bg_traffic.bg_traffic_silver.gtfs_route r
    LEFT JOIN (
        SELECT 
            t.route_id,
            MAX(s.shape_dist_traveled) AS total_distance_km
        FROM bg_traffic.bg_traffic_silver.gtfs_trip t
        JOIN bg_traffic.bg_traffic_silver.gtfs_shape s
            ON t.shape_id = s.shape_id
        GROUP BY t.route_id
    ) d ON r.route_id = d.route_id
) AS source
ON target.route_id = source.route_id
WHEN MATCHED THEN UPDATE SET
    target.route_short_name = source.route_short_name,
    target.route_long_name = source.route_long_name,
    target.route_type_id = source.route_type_id,
    target.route_type_name = source.route_type_name,
    target.total_distance_km = source.total_distance_km

WHEN NOT MATCHED THEN INSERT (
    route_sid,
    route_id,
    route_short_name,
    route_long_name,
    route_type_id,
    route_type_name,
    total_distance_km
)

VALUES (
    (SELECT COALESCE(MAX(route_sid), 0) + 1
        FROM bg_traffic.bg_traffic_gold.dim_gtfs_route),
    source.route_id,
    source.route_short_name,
    source.route_long_name,
    source.route_type_id,
    source.route_type_name,
    source.total_distance_km
);

### Dim_GTFS_Stop

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_gtfs_stop AS target
USING (
    SELECT
        stop_id,
        stop_code,
        stop_name,
        stop_latitude,
        stop_longitude
    FROM bg_traffic.bg_traffic_silver.gtfs_stop
    WHERE stop_id IS NOT NULL
) AS source
ON target.stop_id = source.stop_id
WHEN MATCHED THEN UPDATE SET
    target.stop_code = source.stop_code,
    target.stop_name = source.stop_name,
    target.stop_latitude = source.stop_latitude,
    target.stop_longitude = source.stop_longitude
WHEN NOT MATCHED THEN INSERT (
    stop_sid,
    stop_id,
    stop_code,
    stop_name,
    stop_latitude,
    stop_longitude
)
VALUES (
    (SELECT COALESCE(MAX(stop_sid), 0) + 1
    FROM bg_traffic.bg_traffic_gold.dim_gtfs_stop),
    source.stop_id,
    source.stop_code,
    source.stop_name,
    source.stop_latitude,
    source.stop_longitude
);

### Dim_AirStation

In [0]:
MERGE INTO bg_traffic.bg_traffic_gold.dim_air_station AS target
USING (
    SELECT DISTINCT
        location_id,
        location_name,
        latitude,
        longitude,
        provider_name
    FROM bg_traffic.bg_traffic_silver.openaq_locations
    WHERE location_id IS NOT NULL
) AS source
ON target.location_id = source.location_id
WHEN MATCHED THEN UPDATE SET
    target.location_name = source.location_name,
    target.latitude = source.latitude,
    target.longitude = source.longitude,
    target.provider_name = source.provider_name
WHEN NOT MATCHED THEN INSERT (
    location_sid,
    location_id,
    location_name,
    latitude,
    longitude,
    provider_name
)
VALUES (
    (SELECT COALESCE(MAX(location_sid), 0) + 1
    FROM bg_traffic.bg_traffic_gold.dim_air_station),
    source.location_id,
    source.location_name,
    source.latitude,
    source.longitude,
    source.provider_name
);